# SATAT Indian meal recognition — C1 starter
This notebook trains a small classification head on frozen DINOv2 features. **It does not train a complete thali scanner or establish portion/nutrition accuracy.** Read `PRD.md` and the bundled README first.

Prerequisites: reviewed dataset permissions, labelled crop images with a manifest, and a Google Colab GPU assigned through **Runtime → Change runtime type → GPU**. No dataset, key or trained model is bundled. Use data authorised for this Google account/environment. Colab resources are not guaranteed; use durable checkpoints.


In [ ]:
import sys, subprocess, platform
import torch, numpy, PIL
print({'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'pillow': PIL.__version__})
if torch.cuda.is_available():
    print({'gpu': torch.cuda.get_device_name(0), 'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)})
else:
    print('CPU runtime: source checks and data preparation can run. Assign a GPU before training.')


## Upload the starter archive
Upload `indian-meal-starter.zip` using Colab's Files panel (provided with this PRD handoff). The archive contains source/specification only. Upload the authorised dataset separately into `/content/meal-data`, including its populated `manifest.json`. Do not upload `.env`, unrelated Drive files or service credentials.


In [ ]:
from pathlib import Path
import zipfile, stat, os
archive = Path('/content/indian-meal-starter.zip')
workspace = Path('/content/satat-meal-starter').resolve()
if not archive.is_file():
    raise FileNotFoundError('Upload indian-meal-starter.zip in the Files panel first.')
with zipfile.ZipFile(archive) as bundle:
    if sum(item.file_size for item in bundle.infolist()) > 10 * 1024**2:
        raise ValueError('Unexpected starter size; this archive should contain source only.')
    for item in bundle.infolist():
        target = (workspace / item.filename).resolve()
        if not target.is_relative_to(workspace) or stat.S_ISLNK(item.external_attr >> 16):
            raise ValueError('Unsafe archive entry')
    bundle.extractall(workspace)
os.chdir(workspace)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'ml/indian_meals', '-p', 'test_*.py', '-v'], check=True)


## Durable checkpoints
The next cell asks you to connect Google Drive. Review the notebook before granting access. The run directory below is deliberately under Drive so a Colab restart does not erase checkpoints. If you prefer other authorised durable storage, replace this cell and the output path. Colab's local `/content` storage alone is temporary.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Prepare Khana research pilot (optional)
Use this only for the acknowledged **non-commercial research** experiment. It downloads approximately 6.9 GB into mounted Drive and prepares ten dish categories with a real manifest. It does not qualify production weights, portions or nutrition. The download/import uses CPU; do not allocate a GPU merely for this step. If you use a separate CPU session, restore the source and mount Drive again after switching to GPU; the prepared Drive data persists.

Set `PREPARE_KHANA_RESEARCH=True` after reviewing the publisher terms at https://khana.omkarprabhu.in/. Keep it False when supplying your own production-approved dataset. The ten-class pilot cannot pass the unchanged 30-class release benchmark. Check `preparation-report.json` after import.


In [ ]:
PREPARE_KHANA_RESEARCH = False
if PREPARE_KHANA_RESEARCH:
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Run the Drive mounting cell first.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'gdown==6.5.2'], check=True)
    subprocess.run([sys.executable, 'ml/indian_meals/prepare_khana.py',
        '--archive', '/content/drive/MyDrive/SATAT/research/khana-source/dataset.tar.gz',
        '--output', '/content/drive/MyDrive/SATAT/research/khana-10-v1',
        '--download', '--reuse-prepared', '--research-only', '--acknowledge-source-terms'], check=True)
else:
    print('Research import skipped; provide your own reviewed dataset before validation.')


In [ ]:
MANIFEST = '/content/meal-data/manifest.json'
DATA_ROOT = '/content/meal-data'
OUTPUT = '/content/drive/MyDrive/SATAT/meal-runs/pilot-001'
SCOPE = 'production'  # research-only sources require research; those weights are not product-approved
if globals().get('PREPARE_KHANA_RESEARCH', False):
    MANIFEST = '/content/drive/MyDrive/SATAT/research/khana-10-v1/manifest.json'
    DATA_ROOT = '/content/drive/MyDrive/SATAT/research/khana-10-v1'
    OUTPUT = '/content/drive/MyDrive/SATAT/research/meal-runs/khana-10-001'
    SCOPE = 'research'
EPOCHS = 30
BATCH_SIZE = 32
SEED = 42
RESUME = False
if not Path(MANIFEST).is_file():
    raise FileNotFoundError('Provide the real dataset manifest and authorised crop files first. The PRD example is not a dataset.')
common = ['--manifest', MANIFEST, '--data-root', DATA_ROOT, '--output', OUTPUT, '--scope', SCOPE]
subprocess.run([sys.executable, 'ml/indian_meals/benchmark.py', 'validate', '--manifest', MANIFEST, '--data-root', DATA_ROOT, '--scope', SCOPE], check=True)


## Train using train/validation only
The source revision is pinned; pretrained backbone weights are downloaded from the official DINOv2 publisher and hashed. This cell trains a linear head, saves every epoch and calibrates on validation. Set `RESUME=True` after an interrupted run with unchanged configuration. A small pilot can train successfully yet remain too small to qualify.


In [ ]:
command = [sys.executable, 'ml/indian_meals/train.py', 'train', *common, '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--seed', str(SEED)]
if RESUME:
    command.append('--resume')
subprocess.run(command, check=True)


## Sealed test evaluation
Only run after checkpoint, preprocessing and validation calibration are frozen. This is not a tuning loop. Set `RUN_FINAL_TEST=True` deliberately. If the experiment fails, preserve its report; do not relax counts or remove failed images. Further test-informed development requires a fresh release holdout. C1 success still leaves full scanner qualification false.


In [ ]:
RUN_FINAL_TEST = False
if not RUN_FINAL_TEST:
    raise RuntimeError('Freeze the candidate and calibration, then explicitly enable the sealed test evaluation.')
subprocess.run([sys.executable, 'ml/indian_meals/train.py', 'predict', *common, '--batch-size', str(BATCH_SIZE)], check=True)


## Export small review artifacts
The next cell downloads only reports/metadata, not meal photos or large weights. If an acceptance run failed, its report is still written: use this cell to return the failures too. Keep checkpoints in durable storage. Never mark the PRD's full development milestone complete based only on a synthetic test suite or this crop-classifier report.


In [ ]:
from google.colab import files
report_bundle = Path('/content/satat-meal-review.zip')
with zipfile.ZipFile(report_bundle, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name in ['metadata.json', 'calibration.json', 'predictions.json', 'classifier-report.json']:
        artifact = Path(OUTPUT) / name
        if artifact.is_file():
            bundle.write(artifact, name)
files.download(str(report_bundle))
